# PCA PROFESIONAL — Minería de Datos
# Principal Component Analysis con Dataset Real

Este notebook desarrolla de manera completa el algoritmo:

# PCA — Principal Component Analysis

usando un dataset real de vinos.

---

#  Objetivos de aprendizaje

Al finalizar esta clase podrás:

1. Comprender qué problema resuelve PCA.
2. Entender la relación entre varianza e información.
3. Comprender covarianza y correlación.
4. Entender por qué PCA requiere escalado.
5. Interpretar componentes principales.
6. Analizar explained variance ratio.
7. Interpretar scree plots.
8. Comprender loadings.
9. Aplicar PCA para visualización.
10. Combinar PCA y clustering.
11. Interpretar clusters en espacio reducido.
12. Comprender ventajas y limitaciones de PCA.

---

# IDEA CENTRAL

> PCA NO clasifica.

> PCA NO predice.

> PCA transforma variables para conservar la mayor cantidad de información posible en menos dimensiones.

# 1️⃣ Problema que resuelve PCA

En minería de datos aparecen datasets con:

- muchas variables,
- variables correlacionadas,
- ruido,
- redundancia,
- dificultad de visualización.

Ejemplo:

- 20 variables,
- 50 variables,
- 200 variables.

Esto genera problemas:

| Problema | Consecuencia |
|---|---|
| Alta dimensionalidad | difícil visualizar |
| Variables correlacionadas | redundancia |
| Muchas columnas | modelos más complejos |
| Ruido | pérdida de calidad |
| Curse of dimensionality | dificultad para modelar |

PCA intenta resolver esto reduciendo dimensionalidad.

# 2️⃣ Intuición geométrica

Imagina una nube de puntos en muchas dimensiones.

PCA busca:

# las direcciones donde los datos se dispersan más.

Estas direcciones son los:

# componentes principales.

---

## Componentes principales

- PC1 → máxima varianza.
- PC2 → segunda máxima varianza.
- PC3 → tercera máxima varianza.

Cada componente es ortogonal a los demás.

# 3️⃣ Varianza

PCA se basa en maximizar varianza.

La varianza mide dispersión.

Si una variable tiene mayor varianza:

- contiene más información,
- permite distinguir mejor observaciones.

## Fórmula de varianza

\[
Var(X)=\frac{1}{n}\sum_{i=1}^{n}(x_i-\bar{x})^2
\]

# 4️⃣ Covarianza

PCA utiliza relaciones entre variables.

La covarianza mide cómo cambian juntas dos variables.

## Fórmula de covarianza

\[
Cov(X,Y)=\frac{1}{n}\sum_{i=1}^{n}(x_i-\bar{x})(y_i-\bar{y})
\]

# 5️⃣ Autovectores y autovalores

PCA matemáticamente utiliza:

- eigenvectors → direcciones.
- eigenvalues → cantidad de varianza explicada.

Interpretación intuitiva:

| Concepto | Interpretación |
|---|---|
| Eigenvector | dirección importante |
| Eigenvalue | información contenida |

NO es necesario profundizar en álgebra lineal avanzada para usar PCA correctamente.

# 6️⃣ Importación de librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

# 7️⃣ Dataset real — Wine Dataset

Usaremos un dataset real de vinos.

Cada fila representa un vino.

Cada columna representa propiedades químicas.

In [ ]:
wine = load_wine()

X = pd.DataFrame(
    wine.data,
    columns=wine.feature_names
)

y = wine.target

print("Dimensiones:", X.shape)

X.head()

# 8️⃣ Variables disponibles

In [ ]:
X.columns

# 9️⃣ Estadísticas descriptivas

Antes de PCA debemos entender:

- escalas,
- medias,
- desviaciones,
- rangos.

🔥 IMPORTANTE:

PCA es extremadamente sensible a magnitudes.

In [ ]:
X.describe().T

# 🔟 Problema de escalas

Observa:

- algunas variables tienen valores pequeños,
- otras tienen valores enormes.

Si aplicamos PCA directamente:

# las variables grandes dominarán el análisis.

Por eso debemos escalar.

# 1️⃣1️⃣ PCA SIN escalado

Primero veremos qué ocurre si NO escalamos.

In [ ]:
pca_no_scale = PCA(n_components=2)

X_pca_no_scale = pca_no_scale.fit_transform(X)

explained_no_scale = pca_no_scale.explained_variance_ratio_

print("Varianza explicada PC1:", explained_no_scale[0])
print("Varianza explicada PC2:", explained_no_scale[1])
print("Acumulado:", explained_no_scale.sum())

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(
    ["PC1", "PC2"],
    explained_no_scale
)

plt.title("PCA SIN escalado")
plt.ylabel("Explained Variance")

plt.show()

# Interpretación

Frecuentemente, sin escalado:

- una sola variable domina,
- PC1 explica casi toda la varianza,
- PCA pierde equilibrio.

Esto ocurre porque las magnitudes son diferentes.

# 1️⃣2️⃣ Escalado correcto

Usaremos:

# StandardScaler

Después del escalado:

- media ≈ 0
- desviación ≈ 1

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

scaled_df = pd.DataFrame(
    X_scaled,
    columns=X.columns
)

scaled_df.head()

In [ ]:
scaled_df.describe().T[["mean", "std"]]

# 1️⃣3️⃣ Aplicación correcta de PCA

Ahora sí aplicamos PCA sobre datos escalados.

In [ ]:
pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

print("Dimensiones originales:", X_scaled.shape)
print("Dimensiones reducidas:", X_pca.shape)

# 1️⃣4️⃣ Componentes principales

In [ ]:
pca_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2"]
)

pca_df.head()

# 1️⃣5️⃣ Explained Variance Ratio

🔥 MUY IMPORTANTE

Indica cuánto de la información original conserva cada componente.

In [ ]:
explained = pca.explained_variance_ratio_

print("PC1:", explained[0])
print("PC2:", explained[1])
print("Acumulado:", explained.sum())

# Interpretación de explained variance

Ejemplo:

- PC1 = 0.36
- PC2 = 0.19

Interpretación:

- PC1 conserva 36% de la información.
- PC2 conserva 19%.
- juntas conservan 55%.

Esto significa que:

# dos dimensiones resumen más de la mitad de la información original.

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(
    ["PC1", "PC2"],
    explained
)

plt.title("Explained Variance Ratio")
plt.ylabel("Varianza explicada")

plt.show()

# 1️⃣6️⃣ Varianza acumulada

La varianza acumulada permite decidir:

# cuántos componentes conservar.

In [ ]:
explained_cumulative = np.cumsum(explained)

explained_cumulative

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    range(1, len(explained_cumulative)+1),
    explained_cumulative,
    marker="o"
)

plt.xlabel("Número de componentes")
plt.ylabel("Varianza acumulada")
plt.title("Varianza acumulada")

plt.grid(True)

plt.show()

# 1️⃣7️⃣ Scree Plot

El scree plot muestra:

- cuánta varianza explica cada componente.

In [ ]:
pca_full = PCA()

pca_full.fit(X_scaled)

variance_full = pca_full.explained_variance_ratio_

plt.figure(figsize=(10,5))

plt.plot(
    range(1, len(variance_full)+1),
    variance_full,
    marker="o"
)

plt.xlabel("Número de componentes")
plt.ylabel("Explained Variance Ratio")
plt.title("Scree Plot")

plt.grid(True)

plt.show()

# Interpretación del Scree Plot

Buscamos el:

# "codo"

Después del codo:

- los componentes aportan poca información adicional.

# 1️⃣8️⃣ Visualización PCA

Ahora visualizamos los vinos usando solo:

- PC1
- PC2

In [ ]:
plt.figure(figsize=(9,6))

scatter = plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    c=y
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA - Wine Dataset")

plt.legend(*scatter.legend_elements(), title="Clase")

plt.show()

# Interpretación del gráfico PCA

Cada punto representa un vino.

Si aparecen grupos separados:

- PCA logró conservar estructura importante.

Observa:

- agrupamientos,
- separación,
- dispersión,
- patrones.

# 1️⃣9️⃣ Loadings

Los loadings muestran:

# qué variables originales aportan más a cada componente.

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    columns=["PC1", "PC2"],
    index=X.columns
)

loadings

# Interpretación de loadings

Si una variable tiene valor grande:

- aporta mucho al componente.

Variables positivas:
- empujan en una dirección.

Variables negativas:
- empujan en dirección opuesta.

In [ ]:
plt.figure(figsize=(10,6))

plt.barh(
    loadings.index,
    loadings["PC1"]
)

plt.title("Loadings de PC1")
plt.xlabel("Contribución")
plt.ylabel("Variable")

plt.show()

# 2️⃣0️⃣ Biplot básico

El biplot combina:

- observaciones,
- componentes,
- variables originales.

In [ ]:
plt.figure(figsize=(10,8))

plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    alpha=0.7
)

for i, var in enumerate(X.columns):
    plt.arrow(
        0,
        0,
        loadings.iloc[i,0]*4,
        loadings.iloc[i,1]*4,
        color="red",
        alpha=0.5
    )

    plt.text(
        loadings.iloc[i,0]*4.2,
        loadings.iloc[i,1]*4.2,
        var,
        color="red"
    )

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Biplot PCA")

plt.grid(True)

plt.show()

# 2️⃣1️⃣ PCA + Clustering

🔥 PARTE MÁS IMPORTANTE APLICADA

Combinaremos:

- PCA
- KMeans

para visualizar clusters.

In [ ]:
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(X_pca)

clusters[:10]

In [ ]:
plt.figure(figsize=(9,6))

scatter = plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    c=clusters
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA + KMeans")

plt.legend(*scatter.legend_elements(), title="Cluster")

plt.show()

# Interpretación

KMeans agrupa observaciones similares.

Gracias a PCA:

- podemos visualizar clusters en 2D,
- interpretar grupos,
- detectar separación.

# 2️⃣2️⃣ Método del codo

El método del codo ayuda a decidir:

# cuántos clusters usar.

In [ ]:
inercias = []

k_values = range(1,11)

for k in k_values:

    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    km.fit(X_pca)

    inercias.append(km.inertia_)

plt.figure(figsize=(9,5))

plt.plot(
    k_values,
    inercias,
    marker="o"
)

plt.xlabel("Número de clusters")
plt.ylabel("Inercia")
plt.title("Método del codo")

plt.grid(True)

plt.show()

# 2️⃣3️⃣ Silhouette Score

El silhouette score evalúa:

# qué tan bien separados están los clusters.

In [ ]:
silhouette = silhouette_score(X_pca, clusters)

print("Silhouette score:", silhouette)

# Interpretación del silhouette score

| Valor | Interpretación |
|---|---|
| cercano a 1 | clusters muy separados |
| cercano a 0 | clusters mezclados |
| negativo | mala agrupación |

# 2️⃣4️⃣ Limitaciones de PCA

PCA NO es perfecto.

## Limitaciones

- pierde interpretabilidad,
- puede perder información,
- asume relaciones lineales,
- depende del escalado,
- no funciona bien con variables categóricas puras.

# 2️⃣5️⃣ ¿Cuándo usar PCA?

## Sí usar PCA cuando:

- hay muchas variables,
- existen correlaciones,
- necesitas visualización,
- deseas reducir dimensionalidad.

---

## No usar PCA cuando:

- necesitas interpretabilidad absoluta,
- tienes datasets pequeños,
- las variables son categóricas puras.

# 2️⃣6️⃣ Pipeline profesional

```python
StandardScaler
↓
PCA
↓
Visualización
↓
KMeans
↓
Interpretación
```

# 2️⃣7️⃣ Errores comunes

## ❌ Aplicar PCA sin escalar

Problema:
- una variable domina.

---

## ❌ Interpretar PCA como clasificación

PCA NO clasifica.

---

## ❌ Usar demasiados componentes

Pierde utilidad de reducción.

---

## ❌ Ignorar explained variance

Debes justificar cuánta información conservaste.

# 2️⃣8️⃣ Conclusiones

En este notebook aprendimos:

1. Qué problema resuelve PCA.
2. Qué es varianza.
3. Qué es covarianza.
4. Por qué escalar.
5. Qué son componentes principales.
6. Qué es explained variance.
7. Cómo interpretar scree plots.
8. Cómo interpretar loadings.
9. Cómo visualizar PCA.
10. Cómo combinar PCA y clustering.

---

# IDEA CENTRAL FINAL

> PCA transforma variables para conservar la mayor cantidad de información posible en menos dimensiones.

# 2️⃣9️⃣ Taller final universitario

## Parte A — Conceptos

Responda:

1. ¿Qué problema resuelve PCA?
2. ¿Qué significa explained variance?
3. ¿Por qué debemos escalar?
4. ¿Qué representa PC1?
5. ¿Qué representa PC2?

---

## Parte B — Aplicación

1. Aplique PCA a 2 componentes.
2. Realice scree plot.
3. Calcule explained variance acumulada.
4. Interprete resultados.

---

## Parte C — Visualización

1. Genere scatter PCA.
2. Interprete agrupamientos.
3. Analice separación de clases.

---

## Parte D — Loadings

1. Identifique variables importantes.
2. Explique contribución de PC1.
3. Explique contribución de PC2.

---

## Parte E — Clustering

1. Aplique KMeans.
2. Realice método del codo.
3. Calcule silhouette score.
4. Interprete clusters.

---

## Parte F — Conclusión

Redacte mínimo 10 líneas indicando:

1. Qué información conservó PCA.
2. Qué tan buena fue la reducción.
3. Qué variables fueron importantes.
4. Qué clusters aparecieron.
5. Ventajas y limitaciones de PCA.